In [2]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax as dfx
import optax
import numpy as np
from jax import random
import matplotlib.pyplot as plt
from tqdm import tqdm

# ============================================================================
# CONFIGURATION
# ============================================================================

DATA_PATH = 'spirals.npz'
TRAIN_SAMPLES = 10000
TEST_SAMPLES = 10000
USE_VALIDATION = False  # Enable/disable validation
VALIDATION_SPLIT = 0.2

INPUT_DIM = 3        # [x, y, time]
HIDDEN_DIM = 32
NUM_CLASSES = 10     # Number of alpha classes

NUM_EPOCHS = 20
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
RANDOM_SEED = 42

OUTPUT_FILE = 'alpha_predictions.npy'

# ============================================================================
# DATA LOADING
# ============================================================================

def load_data(filepath, n_train=None, n_test=None):
    """Load spiral data and prepare for classification."""
    print("Loading data...")
    data = np.load(filepath)
    
    xy_train = data['xy_train'][:n_train] if n_train else data['xy_train']
    alpha_train = data['alpha_train'][:n_train] if n_train else data['alpha_train']
    xy_test = data['xy_test'][:n_test] if n_test else data['xy_test']
    
    # Create time dimension (normalized 0 to 1)
    seq_len = xy_train.shape[1]
    time = jnp.linspace(0, 1, seq_len)
    
    # Add time as third dimension: (batch, seq_len, 3)
    print("Adding time dimension...")
    def add_time(xy):
        batch_size = xy.shape[0]
        time_expanded = jnp.tile(time[None, :, None], (batch_size, 1, 1))
        return jnp.concatenate([xy, time_expanded], axis=-1)
    
    train_data = add_time(jnp.array(xy_train))
    test_data = add_time(jnp.array(xy_test))
    
    # Convert alpha to class labels
    print("Converting alpha to class labels...")
    alpha_min, alpha_max = alpha_train.min(), alpha_train.max()
    alpha_classes = ((alpha_train - alpha_min) / (alpha_max - alpha_min) * (NUM_CLASSES - 1))
    alpha_classes = jnp.clip(jnp.round(alpha_classes), 0, NUM_CLASSES - 1).astype(jnp.int32)
    alpha_classes = alpha_classes.squeeze()
    
    print(f"Train shape: {train_data.shape}")
    print(f"Test shape: {test_data.shape}")
    print(f"Alpha classes: {NUM_CLASSES}")
    print(f"Alpha range: [{alpha_min:.2f}, {alpha_max:.2f}]")
    
    return train_data, alpha_classes, test_data, alpha_min, alpha_max, alpha_train

# ============================================================================
# GRU-ODE MODEL
# ============================================================================

class ODEFunc(eqx.Module):
    """ODE function for continuous evolution."""
    mlp: eqx.nn.MLP
    hidden_size: int

    def __init__(self, hidden_size: int, *, key):
        self.hidden_size = hidden_size
        self.mlp = eqx.nn.MLP(
            in_size=hidden_size,
            out_size=hidden_size,
            width_size=hidden_size * 2,
            depth=2,
            activation=jax.nn.softplus,
            key=key,
        )

    def __call__(self, t, h, args):
        h = jnp.reshape(h, (-1,))
        return self.mlp(h)


class GRUCell(eqx.Module):
    """GRU Cell for observation updates."""
    Wz: jnp.ndarray
    Wr: jnp.ndarray
    Wh: jnp.ndarray
    
    def __init__(self, input_size, hidden_size, key):
        key_z, key_r, key_h = random.split(key, 3)
        scale = 1.0 / jnp.sqrt(hidden_size)
        self.Wz = random.normal(key_z, (hidden_size + input_size, hidden_size)) * scale
        self.Wr = random.normal(key_r, (hidden_size + input_size, hidden_size)) * scale
        self.Wh = random.normal(key_h, (hidden_size + input_size, hidden_size)) * scale
    
    def __call__(self, x, h_prev):
        combined = jnp.concatenate([h_prev, x], axis=-1)
        z = jax.nn.sigmoid(combined @ self.Wz)
        r = jax.nn.sigmoid(combined @ self.Wr)
        combined_reset = jnp.concatenate([r * h_prev, x], axis=-1)
        h_prime = jnp.tanh(combined_reset @ self.Wh)
        h = (1 - z) * h_prime + z * h_prev
        return h


class GRUODEClassifier(eqx.Module):
    """GRU-ODE for trajectory classification."""
    ode_func: ODEFunc
    gru_cell: GRUCell
    classifier: eqx.nn.Linear
    hidden_size: int

    def __init__(self, input_size: int, hidden_size: int, num_classes: int, *, key):
        key_ode, key_gru, key_class = random.split(key, 3)
        self.hidden_size = hidden_size
        self.ode_func = ODEFunc(hidden_size, key=key_ode)
        self.gru_cell = GRUCell(input_size, hidden_size, key=key_gru)
        self.classifier = eqx.nn.Linear(hidden_size, num_classes, key=key_class)

    def __call__(self, trajectory):
        """
        trajectory: (seq_len, 3) - [x, y, time]
        returns: (num_classes,) logits
        """
        seq_len = trajectory.shape[0]
        h = jnp.zeros((self.hidden_size,), dtype=jnp.float32)
        
        # Extract time points
        times = trajectory[:, 2]
        
        solver = dfx.Dopri5()
        term = dfx.ODETerm(self.ode_func)
        
        for i in range(seq_len):
            # Evolve hidden state via ODE
            if i > 0:
                t0, t1 = times[i-1], times[i]
                solution = dfx.diffeqsolve(
                    term, solver,
                    t0=t0, t1=t1,
                    dt0=(t1 - t0) / 5.0,
                    y0=h,
                    saveat=dfx.SaveAt(t1=True),
                    max_steps=16,
                )
                h = jnp.reshape(solution.ys, (-1,))
            
            # Update with observation (x, y)
            obs = trajectory[i, :2]  # Just x, y
            h = self.gru_cell(obs, h)
        
        # Classify
        logits = self.classifier(h)
        return logits

# ============================================================================
# TRAINING
# ============================================================================

def loss_fn(model, trajectory, label):
    """Cross-entropy loss."""
    logits = model(trajectory)
    return optax.softmax_cross_entropy_with_integer_labels(logits, label)

@eqx.filter_jit
def train_step(model, opt_state, trajectory, label, optimizer):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, trajectory, label)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

def compute_accuracy(model, data, labels):
    """Compute classification accuracy."""
    correct = 0
    for i in tqdm(range(len(data)), desc="Computing accuracy", leave=False):
        logits = model(data[i])
        pred = jnp.argmax(logits)
        if pred == labels[i]:
            correct += 1
    return correct / len(data)

def train_model(model, train_data, train_labels, val_data, val_labels, 
                num_epochs, batch_size, key, use_validation=False):
    """Training loop."""
    optimizer = optax.adam(LEARNING_RATE)
    opt_state = optimizer.init(eqx.filter(model, eqx.is_array))
    
    num_samples = train_data.shape[0]
    train_losses = []
    val_accs = []
    
    print("Starting training...")
    
    # Epoch progress bar
    for epoch in tqdm(range(num_epochs), desc="Epochs", position=0):
        key, subkey = random.split(key)
        perm = random.permutation(subkey, num_samples)
        
        epoch_loss = 0.0
        num_batches = 0
        
        # Batch progress bar
        batch_iterator = range(0, num_samples, batch_size)
        for i in tqdm(batch_iterator, desc=f"Epoch {epoch+1}/{num_epochs}", position=1, leave=False):
            batch_idx = perm[i:i+batch_size]
            batch_loss = 0.0
            
            for j in batch_idx:
                model, opt_state, loss = train_step(
                    model, opt_state, train_data[j], train_labels[j], optimizer
                )
                batch_loss += loss
            
            batch_loss /= len(batch_idx)
            epoch_loss += batch_loss
            num_batches += 1
        
        avg_loss = epoch_loss / num_batches
        train_losses.append(float(avg_loss))
        
        # Validation accuracy (optional)
        if use_validation:
            val_acc = compute_accuracy(model, val_data, val_labels)
            val_accs.append(val_acc)
            tqdm.write(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}")
        else:
            tqdm.write(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")
    
    return model, train_losses, val_accs

# ============================================================================
# EVALUATION
# ============================================================================

@eqx.filter_jit
def predict_single(model, trajectory):
    logits = model(trajectory)
    return jnp.argmax(logits)

def predict_batch(model, data):
    predictions = []
    for i in range(data.shape[0]):
        pred = predict_single(model, data[i])
        predictions.append(pred)
    return jnp.array(predictions)

# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    # Load data
    train_data, train_labels, test_data, alpha_min, alpha_max, alpha_train_raw = \
        load_data(DATA_PATH, TRAIN_SAMPLES, TEST_SAMPLES)
    
    # Split validation (if enabled)
    if USE_VALIDATION:
        n_train = int(train_data.shape[0] * (1 - VALIDATION_SPLIT))
        val_data = train_data[n_train:]
        val_labels = train_labels[n_train:]
        val_alpha_raw = alpha_train_raw[n_train:]
        train_data = train_data[:n_train]
        train_labels = train_labels[:n_train]
        train_alpha_raw = alpha_train_raw[:n_train]
        
        print(f"\nTraining samples: {len(train_data)}")
        print(f"Validation samples: {len(val_data)}")
    else:
        val_data = None
        val_labels = None
        val_alpha_raw = None
        train_alpha_raw = alpha_train_raw
        print(f"\nTraining samples: {len(train_data)}")
        print("Validation: DISABLED")
    
    # Initialize model
    key = random.PRNGKey(RANDOM_SEED)
    key, model_key = random.split(key)
    
    model = GRUODEClassifier(
        input_size=2,  # x, y (time is used for ODE evolution)
        hidden_size=HIDDEN_DIM,
        num_classes=NUM_CLASSES,
        key=model_key
    )
    
    # Train
    model, train_losses, val_accs = train_model(
        model, train_data, train_labels, val_data, val_labels,
        NUM_EPOCHS, BATCH_SIZE, key, USE_VALIDATION
    )
    
    print("\nTraining complete!")
    
    # Evaluate on training set
    train_predictions = predict_batch(model, train_data)
    train_alpha_predictions = (train_predictions / (NUM_CLASSES - 1)) * (alpha_max - alpha_min) + alpha_min
    
    train_mae = jnp.mean(jnp.abs(train_alpha_predictions - train_alpha_raw.squeeze()))
    train_rmse = jnp.sqrt(jnp.mean((train_alpha_predictions - train_alpha_raw.squeeze()) ** 2))
    
    print(f"\nTraining Set Performance:")
    print(f"MAE: {train_mae:.4f}")
    print(f"RMSE: {train_rmse:.4f}")
    
    # Validation performance (if enabled)
    if USE_VALIDATION:
        val_predictions = predict_batch(model, val_data)
        val_alpha_predictions = (val_predictions / (NUM_CLASSES - 1)) * (alpha_max - alpha_min) + alpha_min
        
        val_acc = compute_accuracy(model, val_data, val_labels)
        val_mae = jnp.mean(jnp.abs(val_alpha_predictions - val_alpha_raw.squeeze()))
        val_rmse = jnp.sqrt(jnp.mean((val_alpha_predictions - val_alpha_raw.squeeze()) ** 2))
        
        print(f"\nValidation Set Performance:")
        print(f"Accuracy: {val_acc:.4f}")
        print(f"MAE: {val_mae:.4f}")
        print(f"RMSE: {val_rmse:.4f}")
    
    # Test predictions
    test_predictions = predict_batch(model, test_data)
    
    # Convert class predictions back to alpha values
    test_alpha_predictions = (test_predictions / (NUM_CLASSES - 1)) * (alpha_max - alpha_min) + alpha_min
    
    # Save as (TEST_SAMPLES, 1) shape
    test_alpha_predictions_reshaped = np.array(test_alpha_predictions).reshape(-1, 1)
    np.save(OUTPUT_FILE, test_alpha_predictions_reshaped)
    print(f"\nTest predictions saved to {OUTPUT_FILE}")
    print(f"Shape: {test_alpha_predictions_reshaped.shape}")
    
    # Plotting
    if USE_VALIDATION:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Training loss
        axes[0, 0].plot(train_losses)
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].set_title('Training Loss')
        axes[0, 0].grid(True)
        
        # Validation accuracy
        axes[0, 1].plot(val_accs)
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].set_title('Validation Accuracy')
        axes[0, 1].grid(True)
        
        # Training predictions vs actual
        axes[1, 0].scatter(train_alpha_raw, train_alpha_predictions, alpha=0.5, s=20)
        min_val = min(train_alpha_raw.min(), train_alpha_predictions.min())
        max_val = max(train_alpha_raw.max(), train_alpha_predictions.max())
        axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
        axes[1, 0].set_xlabel('True Alpha')
        axes[1, 0].set_ylabel('Predicted Alpha')
        axes[1, 0].set_title(f'Training Set (MAE: {train_mae:.4f}, RMSE: {train_rmse:.4f})')
        axes[1, 0].legend()
        axes[1, 0].grid(True)
        
        # Validation predictions vs actual
        axes[1, 1].scatter(val_alpha_raw, val_alpha_predictions, alpha=0.5, s=20)
        min_val = min(val_alpha_raw.min(), val_alpha_predictions.min())
        max_val = max(val_alpha_raw.max(), val_alpha_predictions.max())
        axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
        axes[1, 1].set_xlabel('True Alpha')
        axes[1, 1].set_ylabel('Predicted Alpha')
        axes[1, 1].set_title(f'Validation Set (MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f})')
        axes[1, 1].legend()
        axes[1, 1].grid(True)
        
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Training loss
        axes[0].plot(train_losses)
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training Loss')
        axes[0].grid(True)
        
        # Training predictions vs actual
        axes[1].scatter(train_alpha_raw, train_alpha_predictions, alpha=0.5, s=20)
        min_val = min(train_alpha_raw.min(), train_alpha_predictions.min())
        max_val = max(train_alpha_raw.max(), train_alpha_predictions.max())
        axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect')
        axes[1].set_xlabel('True Alpha')
        axes[1].set_ylabel('Predicted Alpha')
        axes[1].set_title(f'Training Set (MAE: {train_mae:.4f}, RMSE: {train_rmse:.4f})')
        axes[1].legend()
        axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_results.png', dpi=150)
    print("Plot saved to training_results.png")
    plt.show()

AttributeError: partially initialized module 'jax' has no attribute '_src' (most likely due to a circular import)